# MLIR → SPIR-V → WGSL → WebGPU

Author a GPU kernel in MLIR, compile it in this tab, and run it on your GPU.

Every step happens client-side. There is no server and nothing is precompiled:
the MLIR pass pipeline and the SPIR-V→WGSL translator (Tint) are both compiled
to WebAssembly and shipped in the `mlir-python-bindings` wheel, and the dispatch
goes straight to the browser's own WebGPU implementation.

**Requires a WebGPU-capable browser.** Recent Chrome and Edge work; Safari and
Firefox need it enabled. `navigator.gpu` has to exist.

In [ ]:
%%capture

import piplite
await piplite.install('mlir-python-bindings')
await piplite.install('numpy')

In [ ]:
from js import navigator

if not hasattr(navigator, "gpu"):
    raise RuntimeError(
        "navigator.gpu is missing -- this browser has no WebGPU support, "
        "so the dispatch cell below will not run."
    )

adapter = await navigator.gpu.requestAdapter()
device = await adapter.requestDevice()
print("WebGPU adapter acquired")

## 1. The kernel

A 32×32 matmul as a `gpu.func`. Two things here are load-bearing for what follows:

- `spirv.target_env` with the `Shader` capability, `Logical` addressing and the
  `GLSL450` memory model. That is the Vulkan flavour of SPIR-V, which is what
  WebGPU accepts. The `Kernel`/OpenCL flavour used for CPU and Level Zero targets
  will not translate.
- `spirv.entry_point_abi` carries the workgroup size, which becomes
  `@workgroup_size` in the generated WGSL.

In [ ]:
MATMUL = """
module attributes {
  gpu.container_module,
  spirv.target_env = #spirv.target_env<
    #spirv.vce<v1.0, [Shader], [SPV_KHR_storage_buffer_storage_class]>, #spirv.resource_limits<>>
} {
  gpu.module @kernels {
    gpu.func @matmul(%A: memref<32x32xf32>, %B: memref<32x32xf32>, %C: memref<32x32xf32>)
      kernel attributes {spirv.entry_point_abi = #spirv.entry_point_abi<workgroup_size = [8, 8, 1]>} {
      %row = gpu.global_id x
      %col = gpu.global_id y
      %c0 = arith.constant 0 : index
      %c1 = arith.constant 1 : index
      %c32 = arith.constant 32 : index
      %zero = arith.constant 0.0 : f32
      %sum = scf.for %k = %c0 to %c32 step %c1 iter_args(%acc = %zero) -> (f32) {
        %a = memref.load %A[%row, %k] : memref<32x32xf32>
        %b = memref.load %B[%k, %col] : memref<32x32xf32>
        %m = arith.mulf %a, %b : f32
        %n = arith.addf %acc, %m : f32
        scf.yield %n : f32
      }
      memref.store %sum, %C[%row, %col] : memref<32x32xf32>
      gpu.return
    }
  }
}
"""

print(MATMUL)

## 2. Lower to the SPIR-V dialect

`convert-gpu-to-spirv` rewrites the kernel body, then `spirv-lower-abi-attrs`
turns the `memref` arguments into `spirv.GlobalVariable`s with `bind(set,
binding)` decorations — Vulkan requires entry points to be `void(void)`, so
arguments become interface variables. `spirv-update-vce` computes the
`vce_triple` the serializer requires, and `spirv-webgpu-prepare` expands the ops
WebGPU does not allow.

In [ ]:
from mlir.ir import Context, Module
from mlir.passmanager import PassManager

LOWER = (
    "builtin.module("
    "spirv-attach-target{ver=v1.0 caps=Shader "
    "exts=SPV_KHR_storage_buffer_storage_class client_api=Vulkan},"
    "convert-gpu-to-spirv,"
    "spirv.module(spirv-lower-abi-attrs,spirv-update-vce,spirv-webgpu-prepare))"
)

with Context():
    module = Module.parse(MATMUL)
    PassManager.parse(LOWER).run(module.operation)
    lowered = str(module)

# just the spirv.module, the emptied gpu.module below it is noise here
print(lowered[lowered.index("spirv.module"):lowered.index("gpu.module")])

Note the `bind(0, 0)`, `bind(0, 1)`, `bind(0, 2)` on the global variables. Those
become `@group(0) @binding(N)` in WGSL and decide how the host binds buffers, in
kernel argument order.

## 3. Serialize to a SPIR-V binary

`gpu-module-to-binary` calls `spirv::serialize` internally and stores the result
in a `gpu.binary` op, so no extra binding is needed to get the bytes out.

The re-nesting below is a wart. `convert-gpu-to-spirv` hoists the generated
`spirv.module` to the top level as a *sibling* of the emptied `gpu.module`, while
the serializer only looks for one *inside* a `gpu.module`. Upstream's Vulkan
runner pipeline avoids this with `test-convert-to-spirv{nest-in-gpu-module=true}`,
but that pass is test-only and is not in this wheel.

In [ ]:
def nest_spirv_module(asm):
    lines = asm.split("\n")
    start = next(i for i, l in enumerate(lines) if l.startswith("  spirv.module"))
    depth = 0
    end = None
    for i in range(start, len(lines)):
        depth += lines[i].count("{") - lines[i].count("}")
        # A single-line spirv.module balances at i == start, so close there too.
        if depth == 0:
            end = i
            break
    if end is None:
        raise ValueError("unterminated spirv.module")
    spv = lines[start:end + 1]
    rest = lines[:start] + lines[end + 1:]
    g = next(i for i, l in enumerate(rest) if l.startswith("  gpu.module"))
    return "\n".join(rest[:g + 1] + ["  " + l for l in spv] + rest[g + 1:])


def extract_binary(asm):
    """Pull the blob out of gpu.binary. MLIR escapes bytes as \\XX."""
    i = asm.find("gpu.binary")
    if i < 0:
        raise ValueError("no gpu.binary op -- did gpu-module-to-binary run?")
    if asm.find("gpu.binary", i + 1) >= 0:
        raise ValueError("multiple gpu.binary ops; expected one")
    j = asm.index('"', i) + 1
    out = bytearray()
    while True:
        if j >= len(asm):
            raise ValueError("unterminated gpu.binary blob")
        c = asm[j]
        if c == '"':
            break
        if c == "\\":
            # \XX hex pairs, but \" and \\ for those two characters.
            nxt = asm[j + 1]
            if nxt in ('"', "\\"):
                out.append(ord(nxt)); j += 2
            else:
                out.append(int(asm[j + 1:j + 3], 16)); j += 3
        else:
            out.append(ord(c)); j += 1
    return bytes(out)


with Context():
    module = Module.parse(nest_spirv_module(lowered))
    PassManager.parse("builtin.module(gpu-module-to-binary)").run(module.operation)
    spirv = extract_binary(str(module))

magic = int.from_bytes(spirv[:4], "little")
print(f"{len(spirv)} bytes, magic {magic:#010x} (SPIR-V is 0x07230203)")

## 4. SPIR-V → WGSL

`mlir.wgsl` wraps Tint's SPIR-V reader and WGSL writer, compiled into the wheel.
WebGPU only accepts WGSL, so this step is unavoidable — but note it is *only* the
translator. Dawn's runtime is not here; the browser already implements WebGPU.

In [ ]:
from mlir.wgsl import spirv_to_wgsl

wgsl = spirv_to_wgsl(spirv)
print(wgsl)

## 5. Dispatch

Upload A and B, run the shader, read C back. The bindings line up with the
`bind(0, N)` decorations from step 2 without either side being adjusted to fit
the other.

In [ ]:
import numpy as np
from js import Float32Array, Object
from pyodide.ffi import to_js

def js_obj(d):
    """dict -> plain JS object; WebGPU descriptors are plain objects."""
    return to_js(d, dict_converter=Object.fromEntries)

STORAGE, COPY_DST, COPY_SRC, MAP_READ = 0x80, 0x8, 0x4, 0x1
M = N = K = 32
WG = 8

rng = np.random.default_rng(0)
A = rng.standard_normal((M, K), dtype=np.float32)
B = rng.standard_normal((K, N), dtype=np.float32)

def upload(arr):
    buf = device.createBuffer(js_obj({
        "size": arr.nbytes, "usage": STORAGE | COPY_DST, "mappedAtCreation": True}))
    Float32Array.new(buf.getMappedRange()).set(to_js(arr.ravel().tolist()))
    buf.unmap()
    return buf

a_buf, b_buf = upload(A), upload(B)
out_bytes = M * N * 4
c_buf = device.createBuffer(js_obj({"size": out_bytes, "usage": STORAGE | COPY_SRC}))
read_buf = device.createBuffer(js_obj({"size": out_bytes, "usage": COPY_DST | MAP_READ}))

shader = device.createShaderModule(js_obj({"code": wgsl}))
pipeline = device.createComputePipeline(js_obj({
    "layout": "auto",
    "compute": js_obj({"module": shader, "entryPoint": "matmul"}),
}))
bind_group = device.createBindGroup(js_obj({
    "layout": pipeline.getBindGroupLayout(0),
    "entries": to_js([
        js_obj({"binding": i, "resource": js_obj({"buffer": buf})})
        for i, buf in enumerate((a_buf, b_buf, c_buf))
    ]),
}))

encoder = device.createCommandEncoder()
compute = encoder.beginComputePass()
compute.setPipeline(pipeline)
compute.setBindGroup(0, bind_group)
compute.dispatchWorkgroups((M + WG - 1) // WG, (N + WG - 1) // WG, 1)
compute.end()
encoder.copyBufferToBuffer(c_buf, 0, read_buf, 0, out_bytes)
device.queue.submit(to_js([encoder.finish()]))

await read_buf.mapAsync(MAP_READ)
C = np.asarray(Float32Array.new(read_buf.getMappedRange()).to_py(),
               dtype=np.float32).reshape(M, N).copy()
read_buf.unmap()

expected = A @ B
print("max abs err:", float(np.max(np.abs(C - expected))))
print("MATCH" if np.allclose(C, expected, rtol=1e-4, atol=1e-4) else "MISMATCH")

The error is float32 accumulation noise over a K=32 dot product, not a
correctness problem.

## Try your own

Edit `MATMUL` and re-run from step 2. Things worth knowing:

- Kernel arguments must be `memref` of a type WebGPU can store (`f32` and `i32`
  are safe; `f64` is not available).
- Keep the `Shader` capability. Adding capabilities the browser lacks makes Tint
  reject the module, usually with a message about an unsupported SPIR-V feature.
- Buffers bind at `@group(0) @binding(i)` in argument order.
- If `spirv_to_wgsl` raises, the message carries Tint's own diagnostics plus a
  SPIR-V header summary, which is normally enough to see which op it choked on.